<a href="https://colab.research.google.com/github/JLeeq/berlin-marso-hackathon/blob/main/starter_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WarehouseSort — Starter Notebook

This notebook walks through the **state IL pipeline** (main track) end-to-end on the easy level:
install → look at the env → download demos → train a state Diffusion Policy → evaluate.

The policy reads the **privileged low-dim state vector** (proprioception + parcel poses & tag
colors + bin positions). The provided state Diffusion Policy is your starting point — your job is to improve it. *(An optional harder image/rgb track exists too — see
the README.)*

**Requirements:** a CUDA GPU. In Google Colab: *Runtime → Change runtime type → T4 GPU*.

References:
- [ManiSkill 3](https://maniskill.readthedocs.io/en/latest/) — GPU-accelerated robot simulation
- [Diffusion Policy](https://diffusion-policy.cs.columbia.edu) — Chi et al. 2023

### What a solved episode looks like

The scripted policy (used only to generate the demos) sorting parcels into the
color-matched bins — left panel is the scene view, right is the policy's camera:

![easy demo](https://github.com/marso-robotics/berlin-marso-hackathon/raw/main/media/easy_demo.gif)

*(medium = 4 parcels, hard = 6 parcels with bins that may swap sides — see the README.)*

## 1. Install

## 2. Look at the environment

**Easy level**: 2 parcels (1 red-tagged, 1 blue-tagged), 2 color-coded bins, fixed positions.
The observation is the **state vector**: robot proprioception + parcel poses & tag colors + bin
positions & colors. Its size depends on the parcel count, so a state policy is **level-specific**.

## 3. Demonstrations (provided — Kaggle competition data)

The demos (**200 episodes per level**, state for the main track) are the competition data:
[the data tab](https://www.kaggle.com/competitions/marso-hack-berlin-2026-robot-parcel-sorting-challenge/data). You don't need to record any.

- **On Kaggle** (notebook attached to the competition): the data is already mounted under
  `/kaggle/input/` — nothing to set up; the cell below finds it.
- **On Colab / local**: the cell downloads it with `kagglehub`, which needs your Kaggle API
  credentials (one-time). First **join the competition** (Rules → *I Understand and Accept*),
  then get a token: on kaggle.com click your **avatar → Settings → API → Create New API Token**.
  (Ensure you use the legacy method to download `kaggle.json` containing your `username` and `key`). Then set them in the cell —
  the auth lines are at the top, just uncomment and paste your values.

Either way the cell stages the files under `il/demos/<level>/` so the training commands below
work unchanged.

> ⚠️ The demos come from a scripted policy. Using it to collect *data* is fine, but submitting a
> scripted / hard-coded controller (or any policy that reads privileged env state) is
> **disqualified** — your submitted policy must act from the observation.

In [ ]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


### 📈 Watch training live with TensorBoard

Run the two lines below **before** the training cell. `%tensorboard` starts a background
server and embeds a dashboard that **auto-refreshes as training writes metrics** (loss,
eval sort accuracy), so you can keep watching while the training cell runs.


## 4. Train the state Diffusion Policy

[Diffusion Policy](https://diffusion-policy.cs.columbia.edu) (Chi et al. 2023): a plain MLP
behavior cloner fails due to compounding error; DP's action chunking fixes it.

This uses a short run (`total_iters=10000`) to verify the pipeline. **For real training scale up
to `total_iters=30000`+** (~20–40 min on a T4 for easy; more for medium/hard).

> ⚠️ **One model per level.** The state vector's size depends on the parcel count, so a checkpoint
> is specific to its level — train (and submit) a separate one for easy, medium, and hard.

## 5. Evaluate the checkpoint

**Watch the rollout.** `eval.py` prints the metrics above and saves a rollout video (render + policy-camera views). Display it below — with the untrained template the arm mostly flails, which is exactly the gap you're closing.

In [5]:
%cd /content

!git clone https://github.com/marso-robotics/berlin-marso-hackathon.git

%cd /content/berlin-marso-hackathon

/content
fatal: destination path 'berlin-marso-hackathon' already exists and is not an empty directory.
/content/berlin-marso-hackathon


In [6]:
!pip install -q \
    mani-skill==3.0.1 \
    diffusers==0.38.0 \
    gymnasium \
    torch \
    torchvision \
    hydra-core \
    kagglehub

!pip install -e . -q

import os

os.environ["DISPLAY"] = ""
os.environ["PYOPENGL_PLATFORM"] = "egl"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for warehouse_sort (pyproject.toml) ... done


In [7]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
import os
import shutil

drive_demos = "/content/drive/MyDrive/marso-hackathon/demos"
local_demos = "/content/berlin-marso-hackathon/il/demos"

if os.path.isdir(drive_demos):
    shutil.copytree(
        drive_demos,
        local_demos,
        dirs_exist_ok=True,
    )
    print("Drive에서 demonstration 복원 완료")
else:
    print("Drive에 demonstration 백업이 없습니다.")

Drive에서 demonstration 복원 완료


In [9]:
%cd /content/berlin-marso-hackathon

!ls -lh \
    il/demos/easy/trajectory.state.pd_ee_delta_pos.physx_cuda.h5 \
    il/demos/medium/trajectory.state.pd_ee_delta_pos.physx_cuda.h5 \
    il/demos/hard/trajectory.state.pd_ee_delta_pos.physx_cuda.h5

/content/berlin-marso-hackathon
-rw------- 1 root root 15M Sep 14 12:11 il/demos/easy/trajectory.state.pd_ee_delta_pos.physx_cuda.h5
-rw------- 1 root root 74M Sep 14 12:11 il/demos/hard/trajectory.state.pd_ee_delta_pos.physx_cuda.h5
-rw------- 1 root root 37M Sep 14 12:11 il/demos/medium/trajectory.state.pd_ee_delta_pos.physx_cuda.h5


In [10]:
import os
import shutil

local_runs = (
    "/content/berlin-marso-hackathon/"
    "il/baselines/diffusion_policy/runs"
)

drive_runs = (
    "/content/drive/MyDrive/"
    "marso-hackathon/runs"
)

os.makedirs(drive_runs, exist_ok=True)
os.makedirs(os.path.dirname(local_runs), exist_ok=True)

if os.path.isdir(local_runs) and not os.path.islink(local_runs):
    shutil.copytree(local_runs, drive_runs, dirs_exist_ok=True)
    shutil.rmtree(local_runs)

if not os.path.exists(local_runs):
    os.symlink(drive_runs, local_runs)

print("학습 결과 저장 위치:", os.path.realpath(local_runs))

학습 결과 저장 위치: /content/drive/MyDrive/marso-hackathon/runs


In [11]:
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())

CUDA 사용 가능: False


In [13]:
%cd /content/berlin-marso-hackathon

!python il/train.py method=dp demo_dir=easy \
    flags.sim_backend=cpu \
    flags.total_iters=100 \
    flags.eval_freq=100 \
    flags.num_eval_envs=1 \
    flags.num_eval_episodes=1 \
    flags.exp_name=cpu_test

/content/berlin-marso-hackathon
Could not override 'flags.sim_backend'.
To append to your config use +flags.sim_backend=cpu
Key 'sim_backend' is not in struct
    full_key: flags.sim_backend
    object_type=dict

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


In [ ]:
!python eval.py \
    difficulty=medium \
    obs_mode=state \
    policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_medium_10k/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: medium
  num_parcels: 4
  fixed_poses: false
randomization:
  parcel_po

In [ ]:
%cd /content/berlin-marso-hackathon

!python il/train.py method=dp demo_dir=hard \
    flags.total_iters=10000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_state_dp_hard_10k

/content/berlin-marso-hackathon
[il/train] method=dp
[il/train] cwd=/content/berlin-marso-hackathon/il/baselines/diffusion_policy
[il/train] /usr/bin/python3 train.py --demo-path /content/berlin-marso-hackathon/il/demos/hard/trajectory.state.pd_ee_delta_pos.physx_cuda.h5 --env-id WarehouseSort-v1 --control-mode pd_ee_delta_pos --sim-backend gpu --max-episode-steps 200 --total-iters 10000 --batch-size 256 --obs-horizon 2 --act-horizon 8 --pred-horizon 16 --num-eval-envs 8 --num-eval-episodes 16 --eval-freq 5000 --log-freq 1000 --save-freq 10000 --capture-video --exp-name warehouse_state_dp_hard_10k
2026-09-14 10:38:37.507495: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789382317.529298   10814 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:0

In [ ]:
!python eval.py \
    difficulty=hard \
    obs_mode=state \
    policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_hard_10k/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:78: UserWarning: Failed to find Vulkan ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:100: UserWarning: Failed to find glvnd ICD file. This is probably due to an incorrect or partial installation of the NVIDIA driver. SAPIEN will attempt to provide an ICD file anyway but it may not work.
  warn(
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: hard
  num_parcels: 6
  fixed_poses: false
randomization:
  parcel_pose

In [2]:
%cd /content/berlin-marso-hackathon

!python -c "import yaml, omegaconf, hydra; print('Hydra import 성공')"

[Errno 2] No such file or directory: '/content/berlin-marso-hackathon'
/content
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'hydra'


In [ ]:
%cd /content/berlin-marso-hackathon

!python il/train.py method=dp demo_dir=easy \
    flags.sim_backend=cpu \
    flags.total_iters=100 \
    flags.eval_freq=100 \
    flags.num_eval_envs=1 \
    flags.num_eval_episodes=1 \
    flags.exp_name=cpu_test

In [ ]:
%cd /content/berlin-marso-hackathon

!python il/train.py method=dp demo_dir=easy \
    flags.total_iters=30000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_state_dp_easy_30k

/content/berlin-marso-hackathon
Traceback (most recent call last):
  File "/content/berlin-marso-hackathon/il/train.py", line 20, in <module>
    import hydra
  File "/usr/local/lib/python3.12/dist-packages/hydra/__init__.py", line 5, in <module>
  File "/usr/local/lib/python3.12/dist-packages/hydra/utils.py", line 8, in <module>
    import hydra._internal.instantiate._instantiate2
  File "/usr/local/lib/python3.12/dist-packages/hydra/_internal/instantiate/_instantiate2.py", line 9, in <module>
    from omegaconf import OmegaConf, SCMode
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/__init__.py", line 1, in <module>
    from .base import Container, DictKeyType, Node, SCMode, UnionNode
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/base.py", line 11, in <module>
    from ._utils import (
  File "/usr/local/lib/python3.12/dist-packages/omegaconf/_utils.py", line 24, in <module>
    import yaml
  File "/usr/local/lib/python3.12/dist-packages/yaml/__init__.py", line 

In [ ]:
%cd /content/berlin-marso-hackathon

!python eval.py \
    difficulty=easy \
    obs_mode=state \
    policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_easy_30k/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

In [ ]:
%cd /content/berlin-marso-hackathon

!python il/train.py method=dp demo_dir=medium \
    flags.total_iters=30000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_state_dp_medium_30k

In [ ]:
%cd /content/berlin-marso-hackathon

!python eval.py \
    difficulty=medium \
    obs_mode=state \
    policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_medium_30k/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

In [ ]:
%cd /content/berlin-marso-hackathon

!python il/train.py method=dp demo_dir=hard \
    flags.total_iters=30000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_state_dp_hard_30k

In [ ]:
%cd /content/berlin-marso-hackathon

!python eval.py \
    difficulty=hard \
    obs_mode=state \
    policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_hard_30k/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

## 6. Scaling to medium and hard

Medium (4 parcels) and hard (6 parcels) use the **same pipeline** — but because the state vector
is parcel-count-specific you must **train a separate checkpoint per level** (you can't reuse the
easy model). Just change `demo_dir` and `difficulty`:

```bash
python il/train.py method=dp demo_dir=medium flags.total_iters=50000 flags.exp_name=warehouse_state_dp_medium
python eval.py difficulty=medium policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_medium/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

python il/train.py method=dp demo_dir=hard flags.total_iters=60000 flags.exp_name=warehouse_state_dp_hard
python eval.py difficulty=hard policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_hard/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml
```

Tip: a longer `flags.pred_horizon` (e.g. 32) can help on the longer-horizon levels.

## 7. Optional — image-based track (harder)

Want a harder, more realistic challenge? Try the **RGB** track: the policy sees only a
scene-camera image + proprioception — **no privileged state**. The competition data already
includes the rgb demos (staged in §3), and because the image has a fixed shape, **one rgb
checkpoint can run on all levels**. It's a **template that does not yet solve the task** —
getting an image policy to sort is an open problem.

```bash
# train the RGB Diffusion Policy (ResNet18 + SpatialSoftmax encoder)
python il/train.py method=dp_rgb demo_dir=easy flags.exp_name=warehouse_rgb_dp

# evaluate it (obs_mode=rgb, load_dp_rgb)
python eval.py difficulty=easy obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml
```

## 8. Next steps — how to submit

You've trained and evaluated a policy. From here:

1. **Push to medium and hard** (§6) — they carry the most weight (medium 0.3, hard 0.5).
2. **Or bring your own approach** — any policy that implements the
   `act(obs, deterministic=True)` contract works (RL, scripted, transformer, ...).
3. **Package your entry** — you submit a **GitHub repo** containing your policy
   entrypoint, your checkpoint(s), and a `submission.yaml` declaring the levels +
   checkpoint for each.

**Read [SUBMISSION.md](https://github.com/marso-robotics/berlin-marso-hackathon/blob/main/SUBMISSION.md)**
for the full guide: how the code ties together, the policy contract (with a minimal
example), how to declare what gets scored, and exactly how to submit.

> If you're on Colab, `SUBMISSION.md` and `submission.example.yaml` are also in the
> repo you cloned in step 1 — open them from the file browser on the left.


**What you submit (3 things):** your **codebase** (this repo, forked) · a **`submission.yaml`** (per-level checkpoint) · your **checkpoint(s)** plus the **policy entrypoint** `module:function` that loads them into `act(obs)`.

### Ideas to improve the baseline

- **Train longer / more data**: raise `flags.total_iters`; record more demos with `il/gen_demos.py`.
- **Horizons**: `flags.pred_horizon`, `flags.act_horizon`, `flags.obs_horizon`.
- **Eval denoising steps**: `num_inference_steps` in `load_dp` (more = better, slower).
- **Capacity**: `unet_dims`, `diffusion_step_embed_dim`; **optimisation**: `flags.batch_size`, LR.
- **Generalise** to the held-out wider positions / bin-swaps (hard is weighted 0.5).

> ⚠️ If you change an architecture/horizon hyperparameter for training, pass the **same** value to your policy loader (`load_dp(...)`) or the checkpoint won't load.